In [1]:
import os
import sys
import json
import ollama

# Make sure this notebook's own folder is importable, regardless of the kernel's working directory
# (fixes "ModuleNotFoundError: No module named 'scraper'" when the kernel doesn't start in this folder)
_project_dir = r"C:\Users\pc\Desktop\ai-brochure-generator-main"
for _p in (os.getcwd(), _project_dir):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents

In [2]:
# Using a local model via Ollama instead of the Claude API - no API key needed!
# Make sure the Ollama app is running on your machine (it listens on http://localhost:11434)

MODEL = 'gemma4:e4b'  # the model already pulled locally - run `ollama list` in a terminal to check/change
ollama_client = ollama.Client()

try:
    installed = [m['model'] for m in ollama_client.list()['models']]
    if MODEL in installed:
        print(f"Connected to Ollama - using local model: {MODEL}")
    else:
        print(f"Connected to Ollama, but '{MODEL}' isn't pulled yet. Installed models: {installed}\nRun: ollama pull {MODEL}")
except Exception as e:
    print(f"Couldn't reach Ollama at localhost:11434 - make sure the Ollama app/service is running. ({e})")

Connected to Ollama - using local model: gemma4:e4b


In [3]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog/zh',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 'https://blogs.nvidia.com/blog/nvidia-to-acquire-hugging-face/',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp',
 '/Qwen/Qwen3.8-27B',
 '/Qwen/Qwen3.8-Flash-Next',
 '/zai-org/GLM-5.3',
 '/zai-org/GLM-5.3-Flash',
 '/models',
 '/spaces/pollen-robotics/microduck-simulator',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/kulkas2pintu/QWEN_EDIT_IMAGE',
 '/spaces/AimeeBingmouQu/ProtectBirds',
 '/spaces/MiniMaxAI/MiniMax-H3-Turbo-Lora',
 '/spaces',
 '/datasets/rajpurkar/squad',
 '/datasets/stanfordnlp/imdb',
 '/datasets/nyu-mll/glue',
 '/datasets

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog/zh
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
https://blogs.nvidia.com/blog/nvidia-to-acquire-hugging-face/
/spaces
/models
/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
/Qwen/Qwen3.8-27B
/Qwen/Qwen3.8-Flash-Next
/zai-org/GLM-5.3
/zai-org/GLM-5.3-Flash
/models
/spaces/pollen-robotics/microduck-simulator
/spaces/kulkas2pintu/wan555
/spaces/kulkas2pintu/QWEN_EDIT_IMAGE
/spaces/AimeeBingmouQu/ProtectBirds
/spaces/MiniMaxAI/MiniMax-

In [7]:
def select_relevant_links(url):
    response = ollama_client.chat(
        model=MODEL,
        format="json",  # ask Ollama to constrain the reply to valid JSON (replaces the Claude "{" prefill trick)
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
    )
    result = response['message']['content']
    links = json.loads(result)
    return links

In [8]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/blog'},
  {'type': 'company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'product showcase', 'url': 'https://huggingface.co/models'},
  {'type': 'product showcase', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product showcase', 'url': 'https://huggingface.co/spaces'},
  {'type': 'business information', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}]}

In [9]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
×
We are happy to share our intention to join forces with
NVIDIA
.
Read the announcement
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
Updated
4 days ago
•
133k
•
596
Qwen/Qwen3.8-27B
Updated
21 days ago
•
5.74M
•
13.9k
Qwen/Qwen3.8-Flash-Next
Updated
9 days ago
•
351k
•
4.87k
zai-org/GLM-5.3
Updated
about 19 hours ago
•
304k
•
1.7k
zai-org/GLM-5.3-Flash
Updated
about 19 hours ago
•
65

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\n×\nWe are happy to share our intention to join forces with\nNVIDIA\n.\nRead the announcement\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V

In [15]:
def create_brochure(company_name, url):
    response = ollama_client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response['message']['content']
    display(Markdown(result))

In [16]:
create_brochure("HuggingFace", "https://huggingface.co")

# 🚀 Hugging Face: The AI Community Building the Future

**The Collaboration Platform for Machine Learning Innovation**

Hugging Face is the central hub for the rapidly evolving field of Artificial Intelligence. We are dedicated to making advanced machine learning accessible, enabling researchers, developers, and enterprises worldwide to build the next generation of AI applications. Whether you are a hobbyist experimenting with deep learning or a major corporation deploying enterprise-grade AI, Hugging Face provides the tools and community to turn ideas into intelligent products.

***

## 🌐 Platform Offerings: The Core of ML Innovation

Hugging Face provides a comprehensive, unified platform designed to streamline the entire machine learning lifecycle—from research to production.

**Models:**
Browse and utilize a massive repository of over **2 Million+ open-source models**. These models cover advanced capabilities in Natural Language Processing (NLP), Computer Vision, and more, allowing users to explore the latest breakthroughs, including large language models and specialized architectures.

**Datasets:**
Access a curated collection of **500k+ datasets**. These are the foundational raw materials for AI, providing researchers with standardized, high-quality data—such as complex speech transcripts or annotated images—to train and evaluate their models.

**Spaces & Applications:**
Explore over **1 Million+ live AI applications (Spaces)**. These are interactive, runnable demos that allow potential users and collaborators to test models and concepts in action, providing immediate proof-of-concept utility.

**Enterprise & Infrastructure:**
For organizations scaling AI, we offer robust solutions including:
*   **Inference Endpoints:** Reliable, high-performance hosting for deploying models at scale.
*   **Storage Buckets:** Secure cloud storage for all assets.
*   **Enterprise Support:** Dedicated resources for large-scale institutional adoption.

***

## 🧑‍🤝‍🧑 Culture and Community

Our culture is fundamentally built around **open collaboration**. We are more than just a platform—we are a global community.

*   **Community Driven:** Hugging Face fosters an ecosystem where sharing is rewarded. Through forums, GitHub, and our daily papers, members collaborate, critique, and improve models, ensuring that the state-of-the-art remains accessible to all.
*   **Educational Focus:** We are committed to knowledge transfer, offering resources like the **Hugging Face Learn** platform, blogs, and community tutorials, making cutting-edge ML education available regardless of your experience level.
*   **Innovation:** Our deep connections with industry leaders and partners, such as our intent to join forces with **NVIDIA**, solidifies our position at the leading edge of AI technology.

***

## 💼 Careers & Opportunity (For Potential Employees)

We are looking for innovators who are passionate about the frontier of technology. Join a pioneering team that is actively building the infrastructure for the future of AI.

*   **Focus Areas:** Our specialty areas—Machine Learning, NLP, and Deep Learning—require talent in software development, research, and platform engineering.
*   **Growth:** As a specialized Software Development company founded in 2016 and expanding its scope continuously, we offer opportunities to work on high-impact, world-changing projects.
*   **Get Involved:** We encourage prospective employees to explore our job listings and learn more about joining a company defining the global standard for ML collaboration.

***

## 💰 Company Overview (For Investors)

| Detail | Information |
| :--- | :--- |
| **Founded** | 2016 |
| **Industry** | Software Development (Artificial Intelligence) |
| **Specialties** | Machine Learning, Natural Language Processing, Deep Learning |
| **Market Position** | The leading "Home of Machine Learning" platform, facilitating model, dataset, and application collaboration. |
| **Model** | SaaS/Platform Infrastructure (Enterprise Focus) |

**Hugging Face: We empower the architects of tomorrow's intelligent systems.**

In [17]:
def stream_brochure(company_name, url):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    stream = ollama_client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True,
    )
    for chunk in stream:
        response += chunk['message']['content']
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

# 🌐 Hugging Face: Building the Future of AI

**The Collaboration Platform for the Machine Learning Community**

Hugging Face is the central hub powering the rapid advancement of Artificial Intelligence. We are more than just a repository; we are a collaborative ecosystem where the world’s ML developers, researchers, and enterprises meet to build, discover, and deploy the next generation of AI applications.

***

### 🚀 For Customers: Powering Innovation

Whether you are building a groundbreaking research model or launching a consumer-facing application, Hugging Face provides the comprehensive toolkit and massive resources needed to succeed.

**Key Offerings & Assets:**

*   **Model Hub (2M+ Models):** Browse, experiment with, and deploy a vast collection of state-of-the-art models (e.g., Qwen, DeepSeek) for various tasks—from text generation to vision processing.
*   **Dataset Hub (500k+ Datasets):** Access structured, pre-cleaned, and complex datasets (like SQuAD or IMDB) ready for training, dramatically reducing data preparation time.
*   **Spaces (ML Applications):** Instantly deploy and share live, interactive ML demos and applications. Our platform allows users to showcase their work easily, scaling from personal projects to professional demonstrations.
*   **Enterprise Solutions:** With offerings like dedicated Inference Endpoints and Enterprise Support, we provide robust, scalable infrastructure for organizations needing production-grade AI solutions.

**Why Choose Us?** We make the complexity of modern AI accessible, allowing businesses to move from concept to deployed model faster than ever before.

### 💼 For Investors: The Central Infrastructure of AI

Hugging Face stands at the epicenter of the global machine learning economy. Our platform acts as the foundational infrastructure layer for AI development, positioning us for exponential growth and market dominance.

*   **Ecosystem Value:** We have cultivated the largest and most active collaboration platform for ML assets globally.
*   **Strategic Partnerships:** Our commitment to industry leaders, including partnerships like our intention to join forces with **NVIDIA**, solidifies our role in the AI supply chain.
*   **Revenue Diversification:** Our revenue streams span professional services (Enterprise Support), hardware deployment (Inference Providers), and premium tools (Hugging Face PRO), ensuring resilient growth.

### 💡 For Recruits: Join the Vanguard of AI

Are you a researcher, developer, or engineer passionate about pushing the boundaries of artificial intelligence? Hugging Face offers a unique opportunity to work alongside the most talented minds in the field.

**Our Culture:**

*   **Collaboration First:** We believe that the power of AI comes from community. Our culture is built on open access, shared knowledge, and mutual contribution.
*   **Pioneering Spirit:** You will be working on some of the most advanced models and solving real-world problems that define the next era of computing.
*   **Learning Environment:** With dedicated resources like our **Blog, Docs, and Learn** sections, the opportunity to stay at the cutting edge of ML is constant.

**Get Involved:** We provide tools and platforms—from our highly active **Forum** and **Discord** community channels to our GitHub repository—for every level of technical contribution, making it easy to find your place in the ML community.

***

### 🌍 Our Community & Culture

Hugging Face is defined by its vibrant, highly technical, and profoundly collaborative culture. We view AI as a collective achievement.

*   **Community Focus:** The platform is designed *by* and *for* the community. Whether through model sharing, dataset curation, or live Spaces demos, every user is a contributor.
*   **Openness:** We promote open collaboration, allowing the entire ML community to benefit from unlimited public models and datasets.
*   **Global Impact:** Our mission, "The AI community building the future," transcends company boundaries; it is a collective pursuit of human-machine intelligence.

**Ready to build the future with us?** Explore our resources today.

*Browse 2M+ Models • Explore 500k+ Datasets • Build on Live Spaces*

: 